# Klasifikasi Penggunaan dan Tutupan Lahan (Land Use / Land Cover) Jawa Timur
### Citra Sentinel-2A + Algoritma Random Forest + Visualisasi Folium (Google Satellite)

**Mata kuliah:** Penambangan Data Sains (*Data Mining*)

Notebook ini berisi alur lengkap: **pemahaman data → pengumpulan data → deskripsi fitur → eksplorasi data → pemodelan Random Forest → eksperimen → peta hasil klasifikasi**.

**Daftar isi**
1. Data Understanding (tujuan analisis, acuan klasifikasi, wilayah, kelas, band Sentinel-2A)
2. Data Collection (shapefile, penentuan sampel, ekstraksi citra)
3. Deskripsi Fitur dan Rumusnya
4. Eksplorasi Data (EDA)
5. Pemodelan Random Forest (data training dan testing)
6. Eksperimen dan Evaluasi
7. Peta Hasil Klasifikasi (Folium + Google Satellite WMS)
8. Kesimpulan

---
# 1. Data Understanding

## 1.1 Tujuan Analisis *Land Use and Land Classification*
1. **Mengklasifikasikan** tutupan/penggunaan lahan di wilayah Jawa Timur ke dalam 6 kelas (sawah, bangunan, mangrove, lahan hijau, perairan terbuka/laut, danau) berdasarkan nilai reflektansi band Sentinel-2A dan indeks spektralnya.
2. **Memahami karakteristik spektral** tiap kelas lahan, yaitu band dan indeks mana yang paling membedakan satu kelas dari kelas lain.
3. **Membangun dan mengevaluasi model Random Forest** yang dinilai memiliki akurasi lebih baik dibandingkan metode dasar seperti Naive Bayes untuk data spasial multispektral.
4. **Menyajikan peta hasil klasifikasi** menggunakan WMS satelit Google Maps melalui integrasi dengan Folium.

## 1.2 Acuan Klasifikasi Lahan di Indonesia
Acuan standar klasifikasi lahan di Indonesia diatur dalam **SNI 7645-1:2014** tentang *Klasifikasi Penutup Lahan*. Standar ini mengelompokkan penutup lahan menjadi hierarki yang komprehensif, tidak hanya membedakan sawah dan non-sawah, tetapi juga memisahkan daerah bervegetasi, perairan (alami dan buatan), serta lahan terbangun. 
Eksperimen ini menggunakan 6 kelas yang mengacu pada pembagian secara umum pada SNI tersebut:
- Sawah
- Bangunan (Permukiman/Lahan Terbangun)
- Mangrove (Vegetasi Pesisir)
- Lahan Hijau (Hutan/Vegetasi Non-Sawah)
- Perairan Terbuka (Laut)
- Danau (Ranu)

## 1.3 Wilayah Studi dan Eksperimen Kelas
Provinsi **Jawa Timur**. Wilayah ini memiliki tutupan lahan yang beragam mencakup taman nasional, area padat penduduk, hingga kawasan perairan terbuka.

## 1.4 Band pada Sentinel-2A
Sentinel-2A membawa sensor MSI (MultiSpectral Instrument) dengan 13 band. Band yang digunakan dalam analisis ini:
| Band | Nama | Resolusi | Kegunaan |
|---|---|---|---|
| B2 | Blue | 10 m | Penetrasi air, pembeda tanah/vegetasi, bangunan |
| B3 | Green | 10 m | Puncak pantulan vegetasi sehat, indeks air |
| B4 | Red | 10 m | Penyerapan klorofil |
| B5 | Red Edge 1 | 20 m | Kandungan klorofil, kesehatan vegetasi |
| B6 | Red Edge 2 | 20 m | Struktur kanopi vegetasi |
| B7 | Red Edge 3 | 20 m | Struktur kanopi vegetasi |
| B8 | NIR | 10 m | Biomassa vegetasi, pemisah daratan dan air |
| B8A | Narrow NIR | 20 m | Vegetasi, kadar air tanaman |
| B11 | SWIR 1 | 20 m | Kelembapan vegetasi/tanah, area terbangun |
| B12 | SWIR 2 | 20 m | Tanah terbuka, area terbangun |


---
# 2. Deskripsi Fitur dan Rumusnya

Selain 10 band reflektansi asli, fitur turunan berupa 8 indeks spektral digunakan untuk meningkatkan performa model:

1. **NDVI** (Normalized Difference Vegetation Index): $\frac{\rho_{B8}-\rho_{B4}}{\rho_{B8}+\rho_{B4}}$
2. **NDWI** (Normalized Difference Water Index): $\frac{\rho_{B3}-\rho_{B8}}{\rho_{B3}+\rho_{B8}}$
3. **MNDWI** (Modified NDWI): $\frac{\rho_{B3}-\rho_{B11}}{\rho_{B3}+\rho_{B11}}$
4. **NDBI** (Normalized Difference Built-up Index): $\frac{\rho_{B11}-\rho_{B8}}{\rho_{B11}+\rho_{B8}}$
5. **NDRE** (Normalized Difference Red Edge): $\frac{\rho_{B8}-\rho_{B5}}{\rho_{B8}+\rho_{B5}}$
6. **EVI** (Enhanced Vegetation Index): $2.5 \times \frac{\rho_{B8}-\rho_{B4}}{\rho_{B8}+6\rho_{B4}-7.5\rho_{B2}+1}$
7. **SAVI** (Soil Adjusted Vegetation Index): $1.5 \times \frac{\rho_{B8}-\rho_{B4}}{\rho_{B8}+\rho_{B4}+0.5}$
8. **BSI** (Bare Soil Index): $\frac{(\rho_{B11}+\rho_{B4})-(\rho_{B8}+\rho_{B2})}{(\rho_{B11}+\rho_{B4})+(\rho_{B8}+\rho_{B2})}$


---
# 3. Data Collection
Kita memuat data yang telah diekstrak nilai band Sentinel-2A nya ke dalam file CSV (dataset_sentinel2_jatim.csv). Data ini diperoleh dengan menarik nilai citra pada titik poligon shapefile.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Memuat dataset
df = pd.read_csv('dataset_sentinel2_jatim.csv')
df.head()

---
# 4. Eksplorasi Data (EDA) & Jumlah Data Training/Testing
Melihat distribusi jumlah sampel per kelas.

In [ ]:
print("Total Data:", len(df))
print("Jumlah Kelas:", df['kelas'].nunique())

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='kelas', palette='viridis')
plt.title('Jumlah Data per Kelas Lahan')
plt.show()

---
# 5. Pemodelan (Random Forest) dan Eksperimen
Algoritma Random Forest digunakan karena akurasinya secara umum lebih unggul untuk data Remote Sensing yang kompleks jika dibandingkan dengan Naive Bayes.

In [ ]:
FITUR = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 
         'NDVI', 'NDWI', 'MNDWI', 'NDBI', 'NDRE', 'EVI', 'SAVI', 'BSI']

X = df[FITUR]
y = df['kelas_id']
kelas_map = dict(zip(df['kelas_id'], df['kelas']))

# Split Data 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Jumlah Data Training: {len(X_train)}")
print(f"Jumlah Data Testing: {len(X_test)}")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("\nAkurasi Model Random Forest:", accuracy_score(y_test, y_pred))
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred, target_names=[kelas_map[i] for i in sorted(kelas_map.keys())]))

## 5.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[kelas_map[i] for i in sorted(kelas_map.keys())],
            yticklabels=[kelas_map[i] for i in sorted(kelas_map.keys())])
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix - Random Forest')
plt.show()

---
# 6. Peta Hasil Klasifikasi (Folium + WMS Google Satellite)
Titik-titik sampel akan di-plot di atas peta WMS satelit dengan representasi warna berdasarkan hasil prediksi atau kelas aktual.

In [ ]:
# Inisialisasi Peta
m = folium.Map(location=[-7.7, 112.5], zoom_start=8)

# WMS Google Maps Satellite Layer
folium.TileLayer(
    tiles='http://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}',
    attr='Google',
    name='Google Satellite',
    overlay=False,
    control=True
).add_to(m)

colors = {
    1: "orange",    # Sawah
    2: "red",       # Bangunan
    3: "purple",    # Mangrove
    4: "green",     # Lahan Hijau
    5: "darkblue",  # Lautan
    6: "lightblue", # Danau
}

# Prediksi keseluruhan data untuk plotting
df['prediksi'] = rf_model.predict(X)

for idx, row in df.iterrows():
    k_id = row['prediksi']
    n_kelas = kelas_map[k_id]
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        popup=f"Prediksi: {n_kelas}",
        color=colors.get(k_id, "black"),
        fill=True,
        fill_color=colors.get(k_id, "black"),
        fill_opacity=0.8
    ).add_to(m)

folium.LayerControl().add_to(m)
m

---
# 7. Kesimpulan
Melalui eksperimen dengan model Random Forest dan tambahan 6 kelas (mengacu pada SNI), kita dapat melihat peningkatan akurasi serta pemetaan yang lebih jelas menggunakan WMS satelit Google Maps melalui library Folium. Random Forest efektif dalam memisahkan wilayah dengan spektrum mirip seperti Lahan Hijau dan Sawah menggunakan kombinasi fitur seperti B8A, NDVI, dan EVI.